In [5]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import optax
import numpy as np
from functools import partial

# Define the neural network using Flax
class PINN(nn.Module):
    @nn.compact
    def __call__(self, t, x):
        # Stack t and x as inputs
        inputs = jnp.stack([t, x], axis=-1)
        # Define a simple MLP with 3 hidden layers of 20 units each
        for _ in range(3):
            inputs = nn.Dense(features=20)(inputs)
            inputs = nn.tanh(inputs)
        u = nn.Dense(features=1)(inputs)
        return u

# Initialize the model
key = jax.random.PRNGKey(0)
model = PINN()
params = model.init(key, jnp.array([0.0]), jnp.array([0.0]))

# Define the PDE residual function
def pde_residual(params, t, x, nu=0.01):
    # Compute u(t, x)
    u_fn = lambda t, x: model.apply(params, t, x)
    u = u_fn(t, x)
    
    # Compute derivatives using JAX automatic differentiation
    u_t = jax.grad(u_fn, argnums=0)(t, x)
    u_x = jax.grad(u_fn, argnums=1)(t, x)
    u_xx = jax.grad(lambda x: jax.grad(u_fn, argnums=1)(t, x), argnums=0)(x)
    
    # Residual of the Burgers equation
    residual = u_t + u * u_x - nu * u_xx
    return residual

# Define the initial condition
def initial_condition(x):
    return -0.5 * (jnp.sign(x) - 1)

# Define the loss function
def loss_fn(params, t_col, x_col, t_ic, x_ic, t_bc_left, x_bc_left, t_bc_right, x_bc_right):
    # PDE residual loss (collocation points)
    vmap_residual = jax.vmap(partial(pde_residual, params), in_axes=(0, 0))
    res = vmap_residual(t_col, x_col)
    loss_res = jnp.mean(jnp.square(res))

    # Initial condition loss
    vmap_u = jax.vmap(lambda t, x: model.apply(params, t, x), in_axes=(0, 0))
    u_ic_pred = vmap_u(t_ic, x_ic)
    u_ic_true = jax.vmap(initial_condition)(x_ic)
    loss_ic = jnp.mean(jnp.square(u_ic_pred - u_ic_true))

    # Boundary condition loss at x = -1
    u_bc_left_pred = vmap_u(t_bc_left, x_bc_left)
    loss_bc_left = jnp.mean(jnp.square(u_bc_left_pred - 1.0))

    # Boundary condition loss at x = 1
    u_bc_right_pred = vmap_u(t_bc_right, x_bc_right)
    loss_bc_right = jnp.mean(jnp.square(u_bc_right_pred - 0.0))

    # Total loss
    total_loss = loss_res + loss_ic + loss_bc_left + loss_bc_right
    return total_loss

# Generate training data
def generate_data(key, N_col=1000, N_ic=100, N_bc=100, T=1.0):
    key1, key2, key3, key4 = jax.random.split(key, 4)
    
    # Collocation points in the domain
    t_col = jax.random.uniform(key1, (N_col,), minval=0, maxval=T)
    x_col = jax.random.uniform(key2, (N_col,), minval=-1, maxval=1)
    
    # Initial condition points at t = 0
    t_ic = jnp.zeros((N_ic,))
    x_ic = jax.random.uniform(key3, (N_ic,), minval=-1, maxval=1)
    
    # Boundary condition points
    t_bc = jax.random.uniform(key4, (N_bc,), minval=0, maxval=T)
    x_bc_left = -1 * jnp.ones((N_bc,))
    x_bc_right = 1 * jnp.ones((N_bc,))
    
    return t_col, x_col, t_ic, x_ic, t_bc, x_bc_left, t_bc, x_bc_right

# Training step
@jax.jit
def train_step(params, opt_state, t_col, x_col, t_ic, x_ic, t_bc_left, x_bc_left, t_bc_right, x_bc_right, optimizer):
    loss, grads = jax.value_and_grad(loss_fn)(params, t_col, x_col, t_ic, x_ic, t_bc_left, x_bc_left, t_bc_right, x_bc_right)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# Training loop
def train(params, key, optimizer, N_col=1000, N_ic=100, N_bc=100, epochs=10000):
    opt_state = optimizer.init(params)
    
    for epoch in range(epochs):
        key, subkey = jax.random.split(key)
        t_col, x_col, t_ic, x_ic, t_bc, x_bc_left, _, x_bc_right = generate_data(subkey, N_col, N_ic, N_bc)
        params, opt_state, loss = train_step(params, opt_state, t_col, x_col, t_ic, x_ic, t_bc, x_bc_left, t_bc, x_bc_right, optimizer)
        
        if epoch % 1000 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.6f}")
    
    return params

# Set up the optimizer
optimizer = optax.adam(learning_rate=1e-3)

# Train the model
key = jax.random.PRNGKey(1)
trained_params = train(params, key, optimizer)

# Test the trained model
t_test = jnp.array([0.5])
x_test = jnp.array([0.0])
u_pred = model.apply(trained_params, t_test, x_test)
print(f"Predicted u(t=0.5, x=0.0): {u_pred[0]:.6f}")

TypeError: Error interpreting argument to <function train_step at 0x1607ff7f0> as an abstract array. The problematic value is of type <class 'function'> and was passed to the function at path optimizer.init.
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.

In [4]:
# Main execution
if __name__ == "__main__":
    nu = 0.01  # Viscosity coefficient
    T = 1.0    # Time horizon

    # Initialize model and optimizer
    model = PINN()
    key = random.PRNGKey(0)
    params = model.init(key, 0.0, 0.0)['params']
    optimizer = optax.adam(learning_rate=1e-3)

    # Train the PINN
    trained_params = train_pinn(model, params, optimizer, key, nu)
    print("Training completed!")

TypeError: PINN.__init__() missing 2 required positional arguments: 'hidden_dim' and 'num_layers'